# Study 16 — CP-018 / CP-019 Verification: Entity Canonicalization Fixes

Before/after evidence for the 4 CP-018 issues the user found by visually inspecting
`study_15_visual_tour.ipynb`'s entity graph, plus CP-019 (a 5th bug found empirically
while verifying CP-018, not one the user had spotted — a reminder that "these are
examples, not an exhaustive list").

**Before** = `data/db/pathosphere_backup_20260712_163720_pre_cp018.db` (untouched backup)
**After** = `data/db/pathosphere.db` (current, post-fix)

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

notebook_dir = Path.cwd()
while not (notebook_dir / 'data/db/pathosphere.db').exists():
    notebook_dir = notebook_dir.parent
    if notebook_dir == notebook_dir.parent:
        break
import os
os.chdir(notebook_dir)

AFTER = sqlite3.connect('data/db/pathosphere.db')
AFTER.row_factory = sqlite3.Row
BEFORE = sqlite3.connect('data/db/pathosphere_backup_20260712_163720_pre_cp018.db')
BEFORE.row_factory = sqlite3.Row

def q(conn, sql, params=()):
    return pd.DataFrame([dict(r) for r in conn.execute(sql, params).fetchall()])

print('Connected. Before/after DBs ready.')

Connected. Before/after DBs ready.


## 1. France type-conflict fix (CP-018 #1)

`FRANCE` (entity_type=`company`, ALL-CAPS agency-wire artifact) had grabbed Wikidata QID `Q142` before the correctly-typed `France` (location) — "first to arrive wins" made all 62 mentions of the real "France" hang off a node labelled *company*.

In [2]:
sql = '''SELECT id, name, entity_type, wikidata_qid, canonical_entity_id
          FROM entities WHERE name IN ('FRANCE','France') ORDER BY name'''
before = q(BEFORE, sql); before['when'] = 'before'
after = q(AFTER, sql); after['when'] = 'after'
pd.concat([before, after]).sort_values(['name','when'])

,id,name,entity_type,wikidata_qid,canonical_entity_id,when
0,232,FRANCE,company,NaN,14508.0,after
0,232,FRANCE,company,Q142,NaN,before
1,14508,France,location,Q142,NaN,after
1,14508,France,location,NaN,232.0,before


## 2. Intergovernmental orgs mistyped as `company` (CP-018 #2)

`EU`/`NATO` (and other alliances) were tagged `entity_type='company'` via the generic spaCy ORG→company mapping — misleading in any graph/dashboard (shown as businesses).

In [3]:
sql = '''SELECT id, name, entity_type, canonical_name
          FROM entities WHERE name IN ('EU','NATO') ORDER BY name'''
before = q(BEFORE, sql); before['when'] = 'before'
after = q(AFTER, sql); after['when'] = 'after'
pd.concat([before, after]).sort_values(['name','when'])

,id,name,entity_type,canonical_name,when
0,675,EU,organization,European Union,after
0,675,EU,company,Basque,before
1,177,NATO,organization,NATO,after
1,177,NATO,company,NATO,before


## 3. UK/Britain/British/England fragmentation (CP-018 #3)

Four separate `entities` rows referring to the same country, no cross-entity dedup — `canonicalize_location_entities()` now unifies them under one canonical (most-mentioned) node.

In [4]:
sql = '''SELECT id, name, entity_type, canonical_name, canonical_entity_id
          FROM entities WHERE name IN ('UK','Britain','British','England') AND entity_type='location'
          ORDER BY name'''
before = q(BEFORE, sql); before['when'] = 'before'
after = q(AFTER, sql); after['when'] = 'after'
pd.concat([before, after]).sort_values(['name','when'])

,id,name,entity_type,canonical_name,canonical_entity_id,when
0,1801,Britain,location,United Kingdom,1421.0,after
0,1801,Britain,location,United Kingdom,NaN,before
1,1557,British,location,United Kingdom,1421.0,after
1,1557,British,location,Welsh,NaN,before
2,1778,England,location,United Kingdom,1421.0,after
2,1778,England,location,NaN,NaN,before
3,1421,UK,location,United Kingdom,NaN,after
3,1421,UK,location,NaN,1408.0,before


## 4. NER boilerplate noise (CP-018 #4)

`VIDEO` (22 mentions) — UI boilerplate ("WATCH VIDEO") mistagged as an organization by spaCy. `purge_noise_entities()` removes it (and similar terms) outright, `extract_entities` now excludes them at creation time going forward.

In [5]:
before_count = BEFORE.execute("SELECT COUNT(*) FROM entities WHERE lower(name)='video'").fetchone()[0]
after_count = AFTER.execute("SELECT COUNT(*) FROM entities WHERE lower(name)='video'").fetchone()[0]
print(f'VIDEO entity rows — before: {before_count}, after: {after_count}')

VIDEO entity rows — before: 1, after: 0


## 5. CP-019 (bonus) — Wikidata fuzzy-search collision

Found empirically *while verifying* CP-018 (not something the user had flagged): `UK` (58 mentions) had `canonical_entity_id` pointing at an entity named "Ukrainian" with `wikidata_qid='Q8798'` — the **Ukrainian language** (ISO 639 code `uk`), not the country. A pure fuzzy-search collision on a short ambiguous acronym, verified via Wikidata's own P31 (instance-of) claim for Q8798: "Natural language", not "country".

In [6]:
sql = '''SELECT id, name, entity_type, canonical_name, wikidata_qid, canonical_entity_id
          FROM entities WHERE name IN ('UK','Ukrainian','Ukraine') ORDER BY name'''
before = q(BEFORE, sql); before['when'] = 'before'
after = q(AFTER, sql); after['when'] = 'after'
pd.concat([before, after]).sort_values(['name','when'])

,id,name,entity_type,canonical_name,wikidata_qid,canonical_entity_id,when
0,1421,UK,location,United Kingdom,NaN,NaN,after
0,1421,UK,location,NaN,NaN,1408.0,before
1,1320,Ukraine,location,Ukraine,Q212,NaN,after
1,1320,Ukraine,location,Ukraine,Q212,NaN,before
2,1408,Ukrainian,location,Ukraine,NaN,1320.0,after
2,1408,Ukrainian,location,Ukrainian,Q8798,NaN,before


Note the bonus effect: `Ukrainian` (the demonym entity) is now itself merged into `Ukraine` (the country entity, id 1320, correct QID Q212) — `canonicalize_location_entities` groups demonym+country-name pairs together, not just alias variants of an acronym.

## 6. Aggregate impact — entity_type distribution & dedup volume

In [7]:
sql_types = "SELECT entity_type, COUNT(*) as n FROM entities GROUP BY entity_type ORDER BY n DESC"
before_t = q(BEFORE, sql_types).set_index('entity_type').rename(columns={'n':'before'})
after_t = q(AFTER, sql_types).set_index('entity_type').rename(columns={'n':'after'})
type_dist = before_t.join(after_t, how='outer').fillna(0).astype(int)
type_dist['delta'] = type_dist['after'] - type_dist['before']
type_dist

,before,after,delta
entity_type,,,
company,1889,1867,-22
location,2196,2194,-2
organization,0,15,15
other,2729,2723,-6
person,2347,2346,-1


In [8]:
before_alias = BEFORE.execute("SELECT COUNT(*) FROM entities WHERE canonical_entity_id IS NOT NULL").fetchone()[0]
after_alias = AFTER.execute("SELECT COUNT(*) FROM entities WHERE canonical_entity_id IS NOT NULL").fetchone()[0]
before_total = BEFORE.execute("SELECT COUNT(*) FROM entities").fetchone()[0]
after_total = AFTER.execute("SELECT COUNT(*) FROM entities").fetchone()[0]
print(f'Total entity rows      — before: {before_total}, after: {after_total}')
print(f'Rows aliased (has canonical_entity_id) — before: {before_alias}, after: {after_alias}')
print(f'Distinct canonical entities (surface rows minus aliases) — '
      f'before: {before_total - before_alias}, after: {after_total - after_alias}')

Total entity rows      — before: 9161, after: 9145
Rows aliased (has canonical_entity_id) — before: 223, after: 245
Distinct canonical entities (surface rows minus aliases) — before: 8938, after: 8900


## 7. Residual risk audit — ambiguous country/common-word names (CP-019)

Not yet linked at audit time (so not yet corrupted) but flagged as at-risk: common English words that are also country names, a classic Wikidata disambiguation trap (Turkey/bird, Jordan/river-or-person, Chad/personal name, Guinea/rodent, Georgia/US state, Congo, Mali, Niger, Jersey). `link_wikidata` now verifies P31 for these before accepting a match (`AMBIGUOUS_ENTITY_NAMES`), instead of discovering a bad match after the fact like CP-019 was found.

In [9]:
ambiguous = ['Turkey','Georgia','Jordan','Chad','Guinea','Niger','Congo','Mali','Jersey']
sql = f'''SELECT name, entity_type, wikidata_qid, wikidata_checked FROM entities
          WHERE name IN ({','.join('?'*len(ambiguous))}) ORDER BY name'''
q(AFTER, sql, ambiguous)

,name,entity_type,wikidata_qid,wikidata_checked
0,Congo,location,NaN,0
1,Georgia,location,NaN,0
2,Guinea,location,NaN,0
3,Jordan,person,NaN,0
4,Mali,location,NaN,0
5,Niger,location,NaN,0
6,Turkey,location,Q43,1


## Conclusione critica

**Risolto e verificato empiricamente sul DB reale** (non solo unit test): tutti e 4 i punti
di CP-018, più il bug CP-019 trovato durante la verifica. `pathos graph` rieseguito
(77516 link, da 83808 pre-canonicalizzazione — la differenza è deduplica reale, non perdita
di segnale). 53 test nuovi, 494 totali verdi.

**Cosa NON garantisce questo lavoro**: l'utente ha segnalato esplicitamente che i 4 punti
di CP-018 erano *esempi*, non un elenco chiuso. I fix di oggi sono generalizzati dove
possibile (stoplist curata riusabile, verifica P31 su conflitto, verifica P31 proattiva su
9 nomi ambigui noti) — ma restano liste curate, non un rilevatore generale di collisioni
Wikidata. È probabile che esistano altre entità mal risolte non ancora osservate,
specialmente tra le migliaia di entità `other`/`company` mai ispezionate visivamente.

**Raccomandazione**: procedere con Fase 4 Dashboard — non c'è più un blocco noto — ma
trattare la qualità delle entità come un'area di miglioramento continuo, non un problema
chiuso definitivamente.